# LARES — full pipeline on Colab (authors only)

Companion to `LARES_artifact.ipynb`. It does the expensive half of the artifact:

1. download the 14.35 GB preprocessed LANL/OpTC dataset,
2. compile the 1-minute CSV files into graph snapshots,
3. package the compiled test splits into the small bundle the evaluation notebook consumes,
4. optionally retrain a configuration from scratch.

**Reviewers do not need this notebook.** Nothing here is required to reproduce the tables.

Expect hours rather than minutes, and about 32 GB of disk. Two things commonly go wrong:
Colab reclaims idle runtimes, and anonymous MEGA links are bandwidth-capped per IP address,
which Colab's shared addresses often exceed. Setting `MEGA_USERNAME` addresses the second.

---
## 1. Environment check

In [ ]:
import os, platform, subprocess, sys

print('Python :', sys.version.split()[0])
print('OS     :', platform.platform())
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                   '--format=csv,noheader']).decode().strip())
except Exception:
    print('GPU    : none detected (CPU runtime; Path A still works)')
print()
mount = '/content' if os.path.isdir('/content') else '/'
print(subprocess.check_output(['df', '-h', mount]).decode())

---
## 2. Configuration

In [ ]:
import os

REPO_GIT_URL     = 'https://github.com/TristanBilot/nids.git'
REPO_ARCHIVE_URL = ''
REPO_DIR         = '/content/lares'
DOWNLOAD_DIR     = '/content/downloads'

DATA_ROOT = '/content/lanl_optc_datasets'

FULL_DATASET_URL = ('https://mega.nz/file/icYyQLRJ'
                    '#r8aiObb_eJXhhfgNMDhbf_asRU61XGuaB5-UxzYfRfo')
MEGA_USERNAME    = ''      # strongly recommended, see the note above

DATASETS    = ['LANL']     # add 'OPTC' if the session budget allows
EXPERIMENTS = [0, 1, 2, 3]

os.environ['LARES_DATA_ROOT'] = DATA_ROOT
os.makedirs(DATA_ROOT, exist_ok=True)
print('Data root:', DATA_ROOT)

---
## 3. Download helpers

Handles direct HTTPS links, MEGA links and Google Drive links.

MEGA links cannot be fetched with `curl` or `wget`: the file is end-to-end encrypted and
the decryption key lives in the URL fragment after `#`, which is never sent to the server.
`megadl` from `megatools` performs the API call, the transfer and the decryption, so it is
installed on demand here.

> **MEGA quota warning.** Anonymous MEGA links are bandwidth-limited per IP address, and
> Colab exits through shared addresses that are often already over quota. The small
> bundles of section 6 download fine; the 14.35 GB archive frequently aborts. Set
> `MEGA_USERNAME` in section 2 to draw on your own account's quota instead, or mount a
> Google Drive copy and point the URL at that path:
>
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> FULL_DATASET_URL = '/content/drive/MyDrive/lanl_optc_datasets.zip'
> ```

In [ ]:
import glob, os, shutil, subprocess

def sh(cmd, check=True):
    """Run a shell command, streaming its output into the notebook.

    subprocess writes to the kernel's file descriptors, which Colab does not show in the
    cell, so output is piped back and printed here. Without this a failing command raises
    with no visible reason.
    """
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='', flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f'command failed with exit code {code}: {cmd}')
    return code

def _install_megatools():
    if shutil.which('megadl') is None:
        sh('apt-get -qq update', check=False)
        sh('apt-get -qq install -y megatools', check=False)
    return shutil.which('megadl') is not None

def _mega_login():
    """Store MEGA credentials in ~/.megarc so megadl uses the account's quota.

    megatools reads this file on its own, which keeps the password off the command line.
    """
    if not MEGA_USERNAME:
        return
    rc = os.path.expanduser('~/.megarc')
    if os.path.isfile(rc):
        return
    import getpass
    pw = getpass.getpass(f'MEGA password for {MEGA_USERNAME} (not echoed, not logged): ')
    with open(rc, 'w') as f:
        f.write(f'[Login]\nUsername = {MEGA_USERNAME}\nPassword = {pw}\n')
    os.chmod(rc, 0o600)
    print('Credentials written to ~/.megarc')

def download(url, dest_dir=None):
    """Download `url` into `dest_dir` and return the path of the downloaded file."""
    dest_dir = dest_dir or DOWNLOAD_DIR
    if url.startswith('/') or url.startswith('file://'):
        return url.replace('file://', '')

    os.makedirs(dest_dir, exist_ok=True)

    if 'mega.nz' in url:
        if not _install_megatools():
            raise RuntimeError('megatools could not be installed; use a direct URL, '
                               'or a Google Drive copy of the archive')
        _mega_login()
        before = set(glob.glob(os.path.join(dest_dir, '*')))
        sh(f'megadl --path {dest_dir} "{url}"')
        new = set(glob.glob(os.path.join(dest_dir, '*'))) - before
        if not new:
            raise RuntimeError(
                'megadl produced no file. The link is most likely over its per-IP '
                'bandwidth quota: set MEGA_USERNAME in section 2 to download through '
                'your own account, or copy the archive to Google Drive, mount it with '
                "google.colab.drive, and point the URL at that local path.")
        return new.pop()

    if 'drive.google.com' in url:
        sh('pip -q install -U gdown', check=False)
        import gdown
        return gdown.download(url=url, output=dest_dir + '/', fuzzy=True, quiet=False)

    name = url.split('/')[-1].split('?')[0] or 'download.bin'
    out = os.path.join(dest_dir, name)
    sh(f'wget -q --show-progress -c -O "{out}" "{url}"')
    return out

def extract(archive, dest):
    """Extract a .tar.gz / .tgz / .tar / .zip archive into `dest`."""
    os.makedirs(dest, exist_ok=True)
    if archive.endswith(('.tar.gz', '.tgz', '.tar')):
        sh(f'tar -xf "{archive}" -C "{dest}"')
    elif archive.endswith('.zip'):
        sh(f'unzip -q -o "{archive}" -d "{dest}"')
    else:
        raise ValueError(f'unknown archive type: {archive}')
    return dest

print('helpers ready')

---
## 4. Fetch the code

In [ ]:
import os, shutil

def find_repo_root(base, max_depth=3):
    """Locate the folder holding src/main.py inside an extracted archive."""
    base = os.path.abspath(base)
    for dirpath, dirnames, _ in os.walk(base):
        dirnames[:] = [d for d in dirnames if d != '__MACOSX']
        if dirpath[len(base):].count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        if os.path.isfile(os.path.join(dirpath, 'src', 'main.py')):
            return dirpath
    return None

def install_repo_from_archive(archive):
    staging = REPO_DIR.rstrip('/') + '_staging'
    shutil.rmtree(staging, ignore_errors=True)
    extract(archive, staging)
    root = find_repo_root(staging)
    if root is None:
        raise RuntimeError(f'{archive} does not contain a src/main.py')
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    shutil.move(root, REPO_DIR)

if os.path.isfile(os.path.join(REPO_DIR, 'src', 'main.py')):
    print('Repository already present at', REPO_DIR)
elif REPO_GIT_URL:
    sh(f'git clone --depth 1 {REPO_GIT_URL} {REPO_DIR}')
elif REPO_ARCHIVE_URL:
    install_repo_from_archive(download(REPO_ARCHIVE_URL))
else:
    # Last resort: upload a zip of the repository from your machine.
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Set REPO_GIT_URL or REPO_ARCHIVE_URL in section 2.')
    print('No code source configured. Upload a .zip or .tar.gz of the repository '
          '(sources + weights/, about 60 MB):')
    uploaded = files.upload()
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    name = next(iter(uploaded))
    path = os.path.join(DOWNLOAD_DIR, name)
    with open(path, 'wb') as f:
        f.write(uploaded[name])
    install_repo_from_archive(path)

os.chdir(REPO_DIR)
assert os.path.isfile('src/main.py'), 'src/main.py not found'

missing = [f'weights_{d}_inductive_exp{e}.pkl'
           for d in DATASETS for e in EXPERIMENTS
           if not os.path.isfile(f'weights/weights_{d}_inductive_exp{e}.pkl')]
print('Repository:', REPO_DIR)
if missing:
    print('WARNING: missing weight files, Path A cannot run:', missing)
else:
    print('All weights required by DATASETS/EXPERIMENTS are present.')
sh('ls -l weights | head', check=False)

---
## 5. Install dependencies

Colab already ships PyTorch, pandas, scikit-learn, joblib and tqdm, so only PyTorch
Geometric and Weights & Biases are added. The compiled PyG extensions
(`torch_scatter`, `torch_sparse`, `torch_cluster`) listed in the README are **not**
required: no module in `src/` imports them, and installing them from source on Colab
takes tens of minutes.

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)

sh('pip -q install torch_geometric==2.6.1 wandb==0.17.9', check=False)

import torch_geometric
print('torch_geometric:', torch_geometric.__version__)

# Weights & Biases is imported by src/main.py but disabled at runtime.
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

---
## 6. Download the dataset and compile the snapshots

`fetch_full_dataset` unpacks the archive and points `LARES_DATA_ROOT` at it wherever it
landed. `compile_snapshots` then runs `src/datasets.py` once per experiment, which is the
*Compile datasets* section of the README.

In [ ]:
import glob, os

def snapshots_of(dataset, exp):
    return glob.glob(os.path.join(DATA_ROOT, dataset, 'compiled', 'test',
                                  f'{dataset}_exp{exp}', '*.pkl'))

def missing_experiments():
    return [(ds, e) for ds in DATASETS for e in EXPERIMENTS if not snapshots_of(ds, e)]

def fetch_full_dataset():
    """Download and unpack the 14.35 GB preprocessed dataset, and point DATA_ROOT at it."""
    global DATA_ROOT
    if any(os.path.isdir(os.path.join(DATA_ROOT, d, 'preprocessed')) for d in ['LANL', 'OPTC']):
        print('Full dataset already unpacked at', DATA_ROOT)
        return
    parent = os.path.dirname(DATA_ROOT.rstrip('/')) or '/content'
    extract(download(FULL_DATASET_URL), parent)
    for cand in [DATA_ROOT] + glob.glob(os.path.join(parent, '*')):
        if any(os.path.isdir(os.path.join(cand, d, 'preprocessed')) for d in ['LANL', 'OPTC']):
            DATA_ROOT = cand
            os.environ['LARES_DATA_ROOT'] = cand
            print('LARES_DATA_ROOT =', cand)
            return
    raise RuntimeError('the archive did not contain LANL/preprocessed or OPTC/preprocessed')

def compile_snapshots(dataset, exp):
    sh(f'python src/datasets.py --dataset={dataset} --dataset_name={dataset}_exp{exp} '
       f'--inductive_experiment=Exp{exp}')

todo = missing_experiments()
if not todo:
    print('Nothing to do: all requested experiments are already compiled.')
else:
    sh('df -h /content', check=False)
    fetch_full_dataset()
    for ds, e in todo:
        compile_snapshots(ds, e)
    still = missing_experiments()
    assert not still, f'compilation did not produce snapshots for {still}'

for ds in DATASETS:
    print(f'{ds}: ' + ', '.join(f'Exp{e}={len(snapshots_of(ds, e))} snapshots'
                                for e in EXPERIMENTS))

---
## 7. Package the bundle for the evaluation notebook

Packs the compiled test splits into one archive per dataset, rewriting each snapshot as a
gzip-compressed pickle with int32 indices and float32 features. That is lossless, since the
loader casts to those types anyway, and it roughly halves the size before compression.

Upload the archives, then set `BUNDLE_URLS` in section 2 of `LARES_artifact.ipynb`. After
this, neither you nor a reviewer repeats the download and the compile.

In [ ]:
exps = ' '.join(f'exp{e}' for e in EXPERIMENTS)
for ds in DATASETS:
    sh(f'python tools/make_colab_bundle.py --dataset {ds} '
       f'--experiments {exps} --out /content/bundles')
sh('ls -lh /content/bundles', check=False)
print('\nDownload the archives from the Files pane, or copy them to Drive:')
print("  from google.colab import drive; drive.mount('/content/drive')")
print("  !cp /content/bundles/*.tar.gz /content/drive/MyDrive/")

---
## 8. Optional: retrain from scratch

The README commands of *Reproduce experiments -> From training*. The released weights are
ignored, the model trains for 10 epochs, and the best epoch by MCC is reported. Keep
`TRAIN_TARGETS` to one configuration per session.

In [ ]:
import subprocess, re, pandas as pd

LOG_DIR = '/content/run_logs'
os.makedirs(LOG_DIR, exist_ok=True)
TRAIN_TARGETS = [(DATASETS[0], EXPERIMENTS[0])]

for ds, e in TRAIN_TARGETS:
    config = f'{ds}_inductive_exp{e}'
    cmd = f'python src/main.py --config={config}'
    print('$', cmd, flush=True)
    proc = subprocess.run(cmd, shell=True, cwd=REPO_DIR, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    with open(f'{LOG_DIR}/{config}_from_training.log', 'w') as f:
        f.write(proc.stdout)
    print(proc.stdout[-3000:])